# Reproducibility and Experiment Tracking

This notebook fills the bridge between **model evaluation** and **hyperparameter tuning**. Before you optimize anything, you need a workflow that makes runs repeatable, comparable, and recoverable.

We will build that workflow around a tiny classification experiment and answer five practical questions:

1. What changes when you do **not** control randomness?
2. What does setting a **seed** actually fix?
3. How do **deterministic settings** reduce hidden drift?
4. What should every run save for **experiment tracking**?
5. How do you compare models **fairly** instead of accidentally comparing protocols?


## 1. Setup

We keep the notebook configuration in one place so each run can be reproduced and audited later.


In [ ]:
import copy
import hashlib
import importlib.util
import json
import random
import shutil
from pathlib import Path

import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

repo_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
utils_path = repo_root / "src/aiml_notebooks/utils.py"
spec = importlib.util.spec_from_file_location("aiml_notebooks_utils", utils_path)
utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(utils)
count_parameters = utils.count_parameters
get_device = utils.get_device
set_seed = utils.set_seed

CONFIG = {
    # Reproducibility
    "seed": 42,
    "dataset_seed": 7,
    "default_split_seed": 42,
    "comparison_seeds": [11, 23, 37],

    # Data
    "n_samples": 700,
    "noise": 0.24,
    "val_fraction": 0.20,
    "test_fraction": 0.20,
    "batch_size": 64,

    # Model
    "input_dim": 2,
    "hidden_dim": 32,
    "num_classes": 2,
    "weight_decay": 1e-4,

    # Training
    "learning_rate": 1e-2,
    "max_epochs": 10,
    "early_stop_patience": 2,
    "accelerator": "cpu",
}

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:0.4f}")

LOG_ROOT = repo_root / "logs" / "reproducibility_tracking"
LOG_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {repo_root}")
print(f"Run artifacts: {LOG_ROOT}")


### Random Seed and Device

The topic is reproducibility, so we deliberately run the training loop on **CPU**. That keeps runtime low and avoids hardware-specific nondeterminism dominating the lesson.


In [ ]:
set_seed(CONFIG["seed"])
L.seed_everything(CONFIG["seed"], workers=True)

detected_device = get_device(prefer_cpu=True)
compute_device = torch.device("cpu")

print(f"Detected device preference: {detected_device}")
print(f"Notebook training device: {compute_device}")
print(f"Base configuration fingerprint seed: {CONFIG['seed']}")


## 2. What Randomness Looks Like

Different libraries maintain their own random number generators. If you do nothing, those states drift independently as you execute cells.


In [ ]:
def sample_random_state():
    return {
        "python_random": round(random.random(), 6),
        "numpy_random": round(float(np.random.rand()), 6),
        "torch_random": round(float(torch.rand(1).item()), 6),
    }

uncontrolled_draws = pd.DataFrame(
    [sample_random_state() for _ in range(3)],
    index=["draw_1", "draw_2", "draw_3"],
)
uncontrolled_draws


Resetting the same seed rewinds those generators to the same starting point, which is the first building block of reproducibility.


In [ ]:
set_seed(CONFIG["seed"])
first_reset = sample_random_state()

set_seed(CONFIG["seed"])
second_reset = sample_random_state()

reset_comparison = pd.DataFrame(
    [first_reset, second_reset],
    index=["after_reset_a", "after_reset_b"],
)
reset_comparison["matches_first_row"] = reset_comparison.eq(reset_comparison.iloc[0]).all(axis=1)
reset_comparison


A seed is necessary, but it is not the whole story. Training code can still use non-deterministic kernels or worker behavior unless you explicitly tighten the runtime settings.


In [ ]:
def set_reproducible_mode(seed: int, deterministic: bool = True) -> dict:
    set_seed(seed)
    L.seed_everything(seed, workers=True)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = deterministic
        torch.backends.cudnn.benchmark = not deterministic

    torch.use_deterministic_algorithms(deterministic, warn_only=True)

    return {
        "seed": seed,
        "deterministic_algorithms": deterministic,
        "cudnn_deterministic": bool(
            torch.backends.cudnn.is_available() and torch.backends.cudnn.deterministic
        ),
        "cudnn_benchmark": bool(
            torch.backends.cudnn.is_available() and torch.backends.cudnn.benchmark
        ),
    }


reproducible_settings = set_reproducible_mode(CONFIG["seed"], deterministic=True)
reproducible_settings


## 3. A Tiny Benchmark Problem

We will use a noisy two-class moon dataset. It is small enough to train quickly but messy enough that split choice and optimization details still matter.


In [ ]:
features, labels = make_moons(
    n_samples=CONFIG["n_samples"],
    noise=CONFIG["noise"],
    random_state=CONFIG["dataset_seed"],
)

data = pd.DataFrame(features, columns=["x1", "x2"])
data["label"] = labels

print(f"Dataset shape: {data.shape}")
display(data.head())
display(data["label"].value_counts().rename("count").to_frame())


A quick plot makes it clear why this is a real modeling problem rather than a trivial linear separation problem.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(
    data=data,
    x="x1",
    y="x2",
    hue="label",
    palette="Set2",
    alpha=0.8,
    ax=ax,
)
ax.set_title("Noisy Two-Moons Classification Task")
ax.legend(title="class")
plt.show()


We also want the train, validation, and test splits themselves to be reproducible. That means the split seed belongs in the experiment record just as much as the training seed.


In [ ]:
def build_splits(data_frame: pd.DataFrame, split_seed: int):
    X = data_frame[["x1", "x2"]].to_numpy(dtype=np.float32)
    y = data_frame["label"].to_numpy(dtype=np.int64)

    X_temp, X_test, y_temp, y_test = train_test_split(
        X,
        y,
        test_size=CONFIG["test_fraction"],
        random_state=split_seed,
        stratify=y,
    )

    val_ratio = CONFIG["val_fraction"] / (1.0 - CONFIG["test_fraction"])
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp,
        y_temp,
        test_size=val_ratio,
        random_state=split_seed,
        stratify=y_temp,
    )

    train_mean = X_train.mean(axis=0, keepdims=True)
    train_std = X_train.std(axis=0, keepdims=True) + 1e-6

    def scale(array):
        return ((array - train_mean) / train_std).astype(np.float32)

    split_tensors = {
        "train_x": torch.tensor(scale(X_train), dtype=torch.float32),
        "train_y": torch.tensor(y_train, dtype=torch.long),
        "val_x": torch.tensor(scale(X_val), dtype=torch.float32),
        "val_y": torch.tensor(y_val, dtype=torch.long),
        "test_x": torch.tensor(scale(X_test), dtype=torch.float32),
        "test_y": torch.tensor(y_test, dtype=torch.long),
    }

    split_summary = pd.DataFrame(
        [
            {"split": "train", "samples": len(y_train), "positive_rate": y_train.mean()},
            {"split": "val", "samples": len(y_val), "positive_rate": y_val.mean()},
            {"split": "test", "samples": len(y_test), "positive_rate": y_test.mean()},
        ]
    )

    metadata = {
        "split_seed": split_seed,
        "train_feature_mean": train_mean.round(4).tolist(),
        "train_feature_std": train_std.round(4).tolist(),
    }
    return split_tensors, split_summary, metadata


base_splits, base_split_summary, base_split_metadata = build_splits(
    data,
    split_seed=CONFIG["default_split_seed"],
)
display(base_split_summary)
base_split_metadata


## 4. Config Capture

A run is only reproducible if you can reconstruct the exact hyperparameters and protocol that created it. A config fingerprint gives each experimental setup a stable identity.


In [ ]:
def canonicalize_config(config: dict) -> dict:
    return json.loads(json.dumps(config, sort_keys=True))


def config_fingerprint(config: dict) -> str:
    payload = json.dumps(canonicalize_config(config), sort_keys=True).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()[:10]


config_id = config_fingerprint(CONFIG)
print(f"Current config fingerprint: {config_id}")
canonicalize_config(CONFIG)


Besides the config, each run should also save a lightweight manifest: the seeds, the metric snapshot, and where the checkpoint lives.


In [ ]:
def write_json(path: Path, payload: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True))


def load_json(path: Path) -> dict:
    return json.loads(path.read_text())


## 5. A Small Lightning Model

The model is intentionally simple. The notebook is about **experiment hygiene**, not about squeezing the last percentage point out of the dataset.


In [ ]:
class MoonClassifier(L.LightningModule):
    def __init__(
        self,
        input_dim: int = CONFIG["input_dim"],
        hidden_dim: int = CONFIG["hidden_dim"],
        num_classes: int = CONFIG["num_classes"],
        learning_rate: float = CONFIG["learning_rate"],
        weight_decay: float = CONFIG["weight_decay"],
    ):
        super().__init__()
        self.save_hyperparameters()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, x):
        return self.network(x)

    def _shared_step(self, batch, stage: str):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        predictions = logits.argmax(dim=1)
        accuracy = (predictions == y).float().mean()
        self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=stage != "train")
        self.log(f"{stage}_acc", accuracy, on_step=False, on_epoch=True, prog_bar=stage != "train")
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        self._shared_step(batch, "test")

    def configure_optimizers(self):
        return torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.learning_rate,
            weight_decay=self.hparams.weight_decay,
        )


base_model = MoonClassifier()
print(base_model)
print(f"Parameter count: {count_parameters(base_model):,}")


We separate data loading and evaluation helpers so the experiment runner can focus on protocol rather than plumbing.


In [ ]:
def make_loaders(split_tensors: dict, batch_size: int, train_seed: int | None = None):
    train_dataset = TensorDataset(split_tensors["train_x"], split_tensors["train_y"])
    val_dataset = TensorDataset(split_tensors["val_x"], split_tensors["val_y"])
    test_dataset = TensorDataset(split_tensors["test_x"], split_tensors["test_y"])

    generator = None
    if train_seed is not None:
        generator = torch.Generator().manual_seed(train_seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        generator=generator,
    )
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    return train_loader, val_loader, test_loader


@torch.no_grad()
def evaluate_model(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    correct = 0
    total = 0
    for x_batch, y_batch in loader:
        logits = model(x_batch)
        predictions = logits.argmax(dim=1)
        correct += int((predictions == y_batch).sum().item())
        total += int(y_batch.numel())
    return correct / total


## 6. Experiment Runner

This is the notebook's core utility. One function trains a run, writes its artifacts, and returns a compact summary we can compare later.


In [ ]:
def read_epoch_metrics(log_dir: str | Path):
    metrics_path = Path(log_dir) / "metrics.csv"
    metrics = pd.read_csv(metrics_path)
    epoch_metrics = metrics.groupby("epoch", as_index=False).mean(numeric_only=True)
    return metrics, epoch_metrics


def run_experiment(
    run_name: str,
    base_config: dict,
    split_seed: int,
    train_seed: int | None,
    deterministic: bool,
    tags: list[str] | None = None,
) -> dict:
    config = copy.deepcopy(base_config)
    run_dir = LOG_ROOT / run_name
    if run_dir.exists():
        shutil.rmtree(run_dir)

    if train_seed is None:
        torch.use_deterministic_algorithms(False)
        if torch.backends.cudnn.is_available():
            torch.backends.cudnn.deterministic = False
            torch.backends.cudnn.benchmark = True
        reproducibility = {
            "seed": None,
            "deterministic_algorithms": False,
            "cudnn_deterministic": False,
            "cudnn_benchmark": bool(torch.backends.cudnn.is_available()),
        }
    else:
        reproducibility = set_reproducible_mode(train_seed, deterministic=deterministic)

    split_tensors, split_summary, split_metadata = build_splits(data, split_seed=split_seed)
    train_loader, val_loader, test_loader = make_loaders(
        split_tensors,
        batch_size=config["batch_size"],
        train_seed=train_seed,
    )

    logger = CSVLogger(
        save_dir=str(repo_root / "logs"),
        name="reproducibility_tracking",
        version=run_name,
    )
    checkpoint_callback = ModelCheckpoint(
        dirpath=Path(logger.log_dir) / "checkpoints",
        filename="best",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    )
    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=config["early_stop_patience"],
        mode="min",
    )

    model = MoonClassifier(
        input_dim=config["input_dim"],
        hidden_dim=config["hidden_dim"],
        num_classes=config["num_classes"],
        learning_rate=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )

    trainer = L.Trainer(
        max_epochs=config["max_epochs"],
        accelerator=config["accelerator"],
        devices=1,
        logger=logger,
        callbacks=[checkpoint_callback, early_stop],
        deterministic=deterministic,
        enable_progress_bar=False,
        enable_model_summary=False,
        log_every_n_steps=1,
    )

    trainer.fit(model, train_loader, val_loader)
    test_metrics = trainer.test(
        model=None,
        dataloaders=test_loader,
        ckpt_path="best",
        verbose=False,
    )[0]

    _, epoch_metrics = read_epoch_metrics(logger.log_dir)
    best_epoch_row = epoch_metrics.dropna(subset=["val_loss"]).sort_values("val_loss").iloc[0]

    summary = {
        "run_name": run_name,
        "config_id": config_fingerprint(config),
        "split_seed": split_seed,
        "train_seed": train_seed,
        "deterministic": deterministic,
        "hidden_dim": config["hidden_dim"],
        "parameter_count": int(count_parameters(model)),
        "best_epoch": int(best_epoch_row["epoch"]),
        "best_val_loss": float(best_epoch_row["val_loss"]),
        "best_val_acc": float(best_epoch_row["val_acc"]),
        "test_acc": float(test_metrics["test_acc"]),
        "checkpoint_path": str(checkpoint_callback.best_model_path),
        "log_dir": str(logger.log_dir),
        "tags": tags or [],
        "split_metadata": split_metadata,
        "reproducibility": reproducibility,
        "config": config,
    }

    manifest = copy.deepcopy(summary)
    manifest["split_summary"] = split_summary.to_dict(orient="records")
    write_json(Path(logger.log_dir) / "config.json", config)
    write_json(Path(logger.log_dir) / "manifest.json", manifest)
    return summary


## 7. Uncontrolled Runs

First we let the protocol drift: different splits, no explicit training seed, and deterministic algorithms disabled. This is how accidental notebook experimentation usually starts.


In [ ]:
uncontrolled_runs = [
    run_experiment(
        run_name="uncontrolled_split_03",
        base_config=CONFIG,
        split_seed=3,
        train_seed=None,
        deterministic=False,
        tags=["uncontrolled"],
    ),
    run_experiment(
        run_name="uncontrolled_split_17",
        base_config=CONFIG,
        split_seed=17,
        train_seed=None,
        deterministic=False,
        tags=["uncontrolled"],
    ),
    run_experiment(
        run_name="uncontrolled_split_91",
        base_config=CONFIG,
        split_seed=91,
        train_seed=None,
        deterministic=False,
        tags=["uncontrolled"],
    ),
]

pd.DataFrame(uncontrolled_runs)[
    ["run_name", "split_seed", "train_seed", "deterministic", "best_val_acc", "test_acc"]
]


Even in this tiny example, the reported score now depends on multiple moving parts at once. That makes it impossible to tell whether a later change helped or whether you just sampled a kinder run.


In [ ]:
uncontrolled_df = pd.DataFrame(uncontrolled_runs)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(
    data=uncontrolled_df,
    x="run_name",
    y="test_acc",
    palette="crest",
    ax=ax,
)
ax.set_ylim(0.75, 1.0)
ax.set_ylabel("Test accuracy")
ax.set_xlabel("")
ax.set_title("Uncontrolled runs can drift noticeably")
plt.xticks(rotation=15)
plt.show()

uncontrolled_df[["run_name", "best_val_acc", "test_acc"]]


## 8. Split Seed Sensitivity

Now we control the training seed and deterministic mode, but we intentionally vary the **data split**. This isolates how much the partition alone can move the final score.


In [ ]:
split_seed_runs = [
    run_experiment(
        run_name=f"split_seed_{seed}",
        base_config=CONFIG,
        split_seed=seed,
        train_seed=CONFIG["seed"],
        deterministic=True,
        tags=["split_sweep"],
    )
    for seed in [3, 21, 57]
]

split_seed_df = pd.DataFrame(split_seed_runs)
split_seed_df[["run_name", "split_seed", "best_val_acc", "test_acc"]]


The spread you see here is not optimization noise. It is the same model and the same training seed on different train/validation/test partitions.


In [ ]:
split_spread = split_seed_df["test_acc"].max() - split_seed_df["test_acc"].min()
print(f"Test accuracy spread from split choice alone: {split_spread:.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
sns.pointplot(data=split_seed_df, x="split_seed", y="test_acc", color="#1f77b4", ax=ax)
ax.set_title("Changing only the split seed still changes the result")
ax.set_ylabel("Test accuracy")
plt.show()


## 9. Training Seed Sensitivity

Next we freeze the split and vary only the **training seed**. This isolates initialization order and shuffled mini-batches.


In [ ]:
training_seed_runs = [
    run_experiment(
        run_name=f"train_seed_{seed}",
        base_config=CONFIG,
        split_seed=CONFIG["default_split_seed"],
        train_seed=seed,
        deterministic=True,
        tags=["seed_sweep"],
    )
    for seed in [7, 13, 29]
]

training_seed_df = pd.DataFrame(training_seed_runs)
training_seed_df[["run_name", "train_seed", "best_val_acc", "test_acc"]]


This sweep is usually narrower than the split sweep, but it still matters. If you only report one lucky seed, the comparison is fragile.


In [ ]:
training_spread = training_seed_df["test_acc"].max() - training_seed_df["test_acc"].min()
print(f"Test accuracy spread from training seed alone: {training_spread:.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
sns.pointplot(data=training_seed_df, x="train_seed", y="test_acc", color="#d62728", ax=ax)
ax.set_title("Changing only the training seed also changes the result")
ax.set_ylabel("Test accuracy")
plt.show()


## 10. Fully Controlled Repeats

If we fix both seeds and force deterministic execution, repeated runs should collapse to the same result. That is what lets you trust regression checks and ablations.


In [ ]:
repeat_a = run_experiment(
    run_name="repeat_a",
    base_config=CONFIG,
    split_seed=CONFIG["default_split_seed"],
    train_seed=CONFIG["seed"],
    deterministic=True,
    tags=["repeat"],
)
repeat_b = run_experiment(
    run_name="repeat_b",
    base_config=CONFIG,
    split_seed=CONFIG["default_split_seed"],
    train_seed=CONFIG["seed"],
    deterministic=True,
    tags=["repeat"],
)

repeat_df = pd.DataFrame([repeat_a, repeat_b])[
    ["run_name", "split_seed", "train_seed", "best_val_acc", "test_acc", "best_epoch"]
]
repeat_df


We can turn that intuition into a hard check by comparing the metric deltas directly.


In [ ]:
repeat_numeric = repeat_df[["best_val_acc", "test_acc", "best_epoch"]]
repeat_deltas = repeat_numeric.diff().abs().iloc[-1]

print("Absolute deltas between controlled repeats:")
display(repeat_deltas.rename("delta").to_frame())

assert repeat_deltas.max() < 1e-7, "Controlled repeats should match exactly on CPU."
print("Controlled repeat check passed.")


## 11. Local Experiment Tracking Table

Notebook work becomes much easier to reason about once every run lands in a single table with seeds, config IDs, and artifact paths.


In [ ]:
all_runs = uncontrolled_runs + split_seed_runs + training_seed_runs + [repeat_a, repeat_b]
experiment_log = pd.DataFrame(all_runs)
experiment_log["test_acc_pct"] = experiment_log["test_acc"] * 100
experiment_log["protocol"] = experiment_log["tags"].apply(lambda tags: ", ".join(tags))

summary_columns = [
    "run_name",
    "protocol",
    "config_id",
    "split_seed",
    "train_seed",
    "deterministic",
    "hidden_dim",
    "parameter_count",
    "best_epoch",
    "best_val_acc",
    "test_acc_pct",
    "checkpoint_path",
]

summary_path = LOG_ROOT / "experiment_summary.csv"
experiment_log[summary_columns].to_csv(summary_path, index=False)

print(f"Saved summary table to: {summary_path}")
experiment_log[summary_columns].sort_values("test_acc_pct", ascending=False)


Final metrics are useful, but training curves tell you **how** a run behaved. They reveal instability, early overfitting, and whether two runs with the same final score got there in different ways.


In [ ]:
_, repeat_metrics = read_epoch_metrics(repeat_a["log_dir"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(repeat_metrics["epoch"], repeat_metrics["train_loss"], marker="o", label="train")
axes[0].plot(repeat_metrics["epoch"], repeat_metrics["val_loss"], marker="s", label="val")
axes[0].set_title("Loss Curves")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(repeat_metrics["epoch"], repeat_metrics["train_acc"], marker="o", label="train")
axes[1].plot(repeat_metrics["epoch"], repeat_metrics["val_acc"], marker="s", label="val")
axes[1].set_title("Accuracy Curves")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

repeat_metrics[["epoch", "train_loss", "val_loss", "train_acc", "val_acc"]]


## 12. Checkpointing

Good experiment tracking always points to the exact saved weights that produced the headline metric. Otherwise the score in your table and the model on disk can silently diverge.


In [ ]:
restored_model = MoonClassifier.load_from_checkpoint(repeat_a["checkpoint_path"])
controlled_splits, _, _ = build_splits(data, split_seed=repeat_a["split_seed"])
_, _, controlled_test_loader = make_loaders(
    controlled_splits,
    batch_size=repeat_a["config"]["batch_size"],
    train_seed=repeat_a["train_seed"],
)
restored_test_acc = evaluate_model(restored_model, controlled_test_loader)

print(f"Recorded test accuracy: {repeat_a['test_acc']:.6f}")
print(f"Reloaded checkpoint test accuracy: {restored_test_acc:.6f}")

assert abs(restored_test_acc - repeat_a["test_acc"]) < 1e-7
print("Checkpoint verification passed.")


## 13. Fair Model Comparison

Now we compare a smaller and a wider model. The key point is that the **comparison protocol** must be held constant: same split, same number of seeds, same epoch budget, same early stopping rule.


In [ ]:
small_config = {**CONFIG, "hidden_dim": 16}
wide_config = {**CONFIG, "hidden_dim": 64}

small_runs = [
    run_experiment(
        run_name=f"fair_small_seed_{seed}",
        base_config=small_config,
        split_seed=CONFIG["default_split_seed"],
        train_seed=seed,
        deterministic=True,
        tags=["fair", "small"],
    )
    for seed in CONFIG["comparison_seeds"]
]

wide_runs = [
    run_experiment(
        run_name=f"fair_wide_seed_{seed}",
        base_config=wide_config,
        split_seed=CONFIG["default_split_seed"],
        train_seed=seed,
        deterministic=True,
        tags=["fair", "wide"],
    )
    for seed in CONFIG["comparison_seeds"]
]

print("Finished fair comparison sweeps.")


A common mistake is to cherry-pick the best run from each model and call that the result. A better protocol reports the **mean and spread** across the same seed set.


In [ ]:
small_df = pd.DataFrame(small_runs)
wide_df = pd.DataFrame(wide_runs)

fair_summary = pd.DataFrame(
    [
        {
            "model": "small",
            "parameters": int(small_df["parameter_count"].iloc[0]),
            "best_single_run": small_df["test_acc"].max(),
            "mean_test_acc": small_df["test_acc"].mean(),
            "std_test_acc": small_df["test_acc"].std(ddof=0),
        },
        {
            "model": "wide",
            "parameters": int(wide_df["parameter_count"].iloc[0]),
            "best_single_run": wide_df["test_acc"].max(),
            "mean_test_acc": wide_df["test_acc"].mean(),
            "std_test_acc": wide_df["test_acc"].std(ddof=0),
        },
    ]
)

display(fair_summary)

best_gap = fair_summary.loc[fair_summary["model"] == "wide", "best_single_run"].iloc[0] - fair_summary.loc[
    fair_summary["model"] == "small", "best_single_run"
].iloc[0]
mean_gap = fair_summary.loc[fair_summary["model"] == "wide", "mean_test_acc"].iloc[0] - fair_summary.loc[
    fair_summary["model"] == "small", "mean_test_acc"
].iloc[0]

print(f"Gap if you cherry-pick the best run: {best_gap:.4f}")
print(f"Gap if you compare the mean over identical seed sweeps: {mean_gap:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(
    fair_summary["model"],
    fair_summary["mean_test_acc"],
    yerr=fair_summary["std_test_acc"],
    capsize=6,
    color=["#4c78a8", "#f58518"],
)
ax.set_ylim(0.75, 1.0)
ax.set_ylabel("Mean test accuracy")
ax.set_title("Fair comparison = same protocol, then compare means")
plt.show()


## 14. Inspect a Saved Manifest

A good run record should be understandable even if you revisit it weeks later without remembering the notebook state.


In [ ]:
example_manifest = load_json(Path(repeat_a["log_dir"]) / "manifest.json")

manifest_preview = {
    "run_name": example_manifest["run_name"],
    "config_id": example_manifest["config_id"],
    "split_seed": example_manifest["split_seed"],
    "train_seed": example_manifest["train_seed"],
    "best_epoch": example_manifest["best_epoch"],
    "test_acc": example_manifest["test_acc"],
    "checkpoint_path": example_manifest["checkpoint_path"],
    "deterministic": example_manifest["deterministic"],
}

display(pd.Series(manifest_preview))
display(pd.Series(example_manifest["reproducibility"]).rename("reproducibility"))


## 15. Key Takeaways

Reproducibility is not just about setting `seed=42`. It is a full protocol that makes your comparisons defensible.


In [ ]:
takeaways = [
    "Seeds control RNG state, but deterministic runtime settings reduce additional hidden drift.",
    "The split seed belongs in the experiment record because partition choice can move metrics by itself.",
    "A config fingerprint makes it harder to confuse similar-looking runs.",
    "Checkpoint paths should live next to the reported metrics so you can reload exactly what you measured.",
    "Fair comparison means same data split, same seed budget, same stopping rule, and the same metric definition.",
]

for item in takeaways:
    print(f"- {item}")
